In [0]:
%run "../../commons/commons_imports"

In [0]:
df_aluno_silver = read(
    base_path=SILVER_PATH,
    table_name=TS_ALUNO,
    format="delta"
).filter(col('CO_MUNICIPIO').isNull())

In [0]:
df_municipio_silver = read(
    base_path=SILVER_PATH,
    table_name=TS_MUNICIPIO,
    format="delta"
)

In [0]:
df_estado_silver = read(
    base_path=SILVER_PATH,
    table_name=TS_ESTADO,
    format="delta"
)

In [0]:
# =====================================================
# Join com a dimensão de Município
#
# Objetivo:
# Enriquecer o dataset de alunos com indicadores
# agregados do município para utilização em análises
# e modelos de Machine Learning.
# =====================================================


df_aluno_municipio_join = (
    df_aluno_silver.alias("a")
    .join(
        df_municipio_silver.alias("m"),
        on=[col("a.ANO_REFERENCIA") == col("m.ANO_REFERENCIA"),
            col("a.CO_MUNICIPIO") == col("m.CO_MUNICIPIO"),
            col("a.TP_DEPENDENCIA") == col("m.ID_TIPO_REDE")],
        how="left")
    .select(
        # =====================================================
        # Todas as colunas do aluno
        # =====================================================
        "a.*",
        # =====================================================
        # Indicadores municipais
        # =====================================================
        col("m.PC_ALUNO_ALFABETIZADO"),
        col("m.VL_MEDIA_LP"),
        col("m.FAIXA_ALFABETIZACAO"),
        col("m.FAIXA_MEDIA_LP")
    )
)

In [0]:

# =====================================================
# Join com a dimensão de Estado
#
# Objetivo:
# Enriquecer o dataset de alunos com indicadores
# agregados do estado para utilização em análises
# e modelos de Machine Learning.
# =====================================================
df_aluno_final_join = (
    df_aluno_municipio_join.alias("a")
    .join(
        df_estado_silver.alias("e"),
        on=[col("a.ANO_REFERENCIA") == col("e.ANO_REFERENCIA"),
            col("a.CO_UF") == col("e.CO_UF"),
            col("a.TP_DEPENDENCIA") == col("e.ID_TIPO_REDE")],
        how="left"
    )
    .select(
        # =====================================================
        # Todas as colunas do aluno enriquecidas com os
        # indicadores municipais
        # =====================================================
        "a.*",
        # =====================================================
        # Indicadores estaduais
        # =====================================================
        col("e.PC_ALUNO_ALFABETIZADO").alias("PC_ALUNO_ALFABETIZADO_ESTADO"),
        col("e.VL_MEDIA_LP").alias("VL_MEDIA_LP_ESTADO")))

In [0]:
df_ml = (

    df_aluno_final_join

    # =====================================================
    # Diferença entre a proficiência do aluno e a média do
    # município
    # =====================================================

    .withColumn(
        "DIF_MEDIA_MUNICIPIO",
        round(
            col("VL_PROFICIENCIA_LP") -
            col("VL_MEDIA_LP"),
            2
        )
    )

    # =====================================================
    # Diferença entre a proficiência do aluno e a média do
    # estado
    # =====================================================

    .withColumn(
        "DIF_MEDIA_ESTADO",
        round(
            col("VL_PROFICIENCIA_LP") -
            col("VL_MEDIA_LP_ESTADO"),
            2
        )
    )

    # =====================================================
    # Indica se o aluno possui proficiência maior ou igual
    # à média do município
    # =====================================================

    .withColumn(
        "IN_ACIMA_MEDIA_MUNICIPIO",
        when(
            col("VL_PROFICIENCIA_LP") >= col("VL_MEDIA_LP"),
            1
        ).otherwise(0)
    )

    # =====================================================
    # Indica se o aluno possui proficiência maior ou igual
    # à média do estado
    # =====================================================

    .withColumn(
        "IN_ACIMA_MEDIA_ESTADO",
        when(
            col("VL_PROFICIENCIA_LP") >= col("VL_MEDIA_LP_ESTADO"),
            1
        ).otherwise(0)
    )

    # =====================================================
    # Diferença entre o percentual de alfabetização do
    # município e do estado
    # =====================================================

    .withColumn(
        "DIF_ALFABETIZACAO_MUNICIPIO",
        round(
            col("PC_ALUNO_ALFABETIZADO_ESTADO") -
            col("PC_ALUNO_ALFABETIZADO"),
            2
        )
    )

    # =====================================================
    # Classifica o desempenho do aluno em relação à média
    # estadual
    # =====================================================

    .withColumn(
        "DESEMPENHO_RELATIVO",
        when(
            col("DIF_MEDIA_ESTADO") >= 30,
            "Muito Acima"
        )
        .when(
            col("DIF_MEDIA_ESTADO") >= 10,
            "Acima"
        )
        .when(
            col("DIF_MEDIA_ESTADO") >= -10,
            "Na Média"
        )
        .when(
            col("DIF_MEDIA_ESTADO") >= -30,
            "Abaixo"
        )
        .otherwise("Muito Abaixo")
    )

    # =====================================================
    # Variável alvo (Target) utilizada em modelos de
    # Machine Learning para classificação
    # =====================================================

    .withColumn(
        "TARGET",
        col("IN_ALFABETIZADO")
    )
    # =====================================================
    # Indica se o aluno participou efetivamente da avaliação
    # =====================================================

    .withColumn(
        "IN_PROVA_VALIDA",
        when(
            (col("IN_PRESENCA_LP") == 1) &
            (col("IN_PREENCHIMENTO_LP") == 1),
            1
        ).otherwise(0)
    )

    # =====================================================
    # Indica se existe proficiência calculada para o aluno
    # =====================================================

    .withColumn(
        "IN_POSSUI_PROFICIENCIA",
        when(
            col("VL_PROFICIENCIA_LP").isNotNull(),
            1
        ).otherwise(0)
    )
)

In [0]:
write_delta(
    df=df_ml,
    base_path=GOLD_PATH,
    table_name=FT_MACHINE_LEARNING,
    write_mode="overwrite"
)